In [ ]:
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
from train import train


In [ ]:
def generate_sine_wave_data(num_samples=100, noise=0.1):
    """
    Generate sine wave data with noise.
    """
    # 选择一个随机的起始点
    start = np.random.uniform(0, 2 * np.pi)
    # 生成均匀分布的x值
    x = np.linspace(start, start + 2 * np.pi, num_samples)
    y = np.sin(x) + np.random.normal(0, noise, num_samples)
    return x, y

In [3]:
# 生成随机噪声序列
def generate_random_noise(num_samples=100, noise=0.1):
    """
    Generate random noise data.
    """
    x = np.linspace(0, 2 * np.pi, num_samples)
    y = np.random.normal(0, noise, num_samples)
    return x, y

In [ ]:
# 生成数据
def generate_sequences(n_samples_per_class, sequence_length):
    """
    Generate sequences of sine waves and random noise.
    """
    sequences_wave = [
        generate_sine_wave_data(sequence_length)[1] for _ in range(n_samples_per_class)
    ]
    sequences_noise = [
        generate_random_noise(sequence_length)[1] for _ in range(n_samples_per_class)
    ]
    sequences = np.array(sequences_wave + sequences_noise)
    labels = np.array([0] * n_samples_per_class + [1] * n_samples_per_class)
    return sequences, labels


# 数据处理
n_samples_per_class = 1000
sequence_length = 100
sequences, labels = generate_sequences(n_samples_per_class, sequence_length)

In [5]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(sequences, labels, test_size=0.2)

# 转换为PyTorch张量
X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(2)
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
X_test = torch.tensor(X_test, dtype=torch.float32).unsqueeze(2)
y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

In [6]:
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
# 构建RNN模型
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_classes = num_classes
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        output, _ = self.rnn(x)
        # output的形状是 (N, L, H)，其中N是批大小，L是序列长度，H是隐藏层大小
        # 输出序列的最后1项时 (N, 1, H)
        last_output = output[:, -1]
        # 分类器将输出 (N, 1, num_classes)
        out = self.fc(last_output)
        # 最后的输出是(N, num_classes)
        return out.view(-1, self.num_classes)

In [ ]:
# 初始化模型、损失函数和优化器
input_size = 1
hidden_size = 32
num_classes = 1
epochs = 1000
model = SimpleRNN(input_size, hidden_size, num_classes)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
train_loss, val_loss = train(
    model, train_loader, test_loader, loss_fn, optimizer, epochs
)

In [ ]:
with torch.inference_mode():
    outputs = model(X_test)
    predicted = torch.sigmoid(outputs).round()
    # 计算准确率
    accuracy = (predicted == y_test).float().mean()
    print(f"Accuracy: {accuracy.item() * 100:.2f}%")

Accuracy: 100.00%
